# Chapter 03: GroupBy and Aggregation

The **split-apply-combine** pattern is one of the most powerful concepts in data analysis. Pandas implements it through the `.groupby()` method: split the data into groups based on one or more keys, apply a function to each group, then combine the results.

In this notebook we will cover:

- The GroupBy concept
- `.groupby()` basics
- `.agg()` with single and multiple functions
- `.transform()` vs `.agg()`
- Multi-column groupby
- `.describe()` on groups
- Named aggregation with `pd.NamedAgg`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
tips = pd.read_csv('data/tips.csv')
tips.head()

## The GroupBy Object

Calling `.groupby()` does not compute anything immediately — it returns a lazy `GroupBy` object that stores the grouping information.

In [ ]:
grouped = tips.groupby('day')
print(type(grouped))
print(f"Number of groups: {grouped.ngroups}")
print(f"Group keys: {list(grouped.groups.keys())}")

In [ ]:
# See which rows belong to each group
grouped.groups

In [ ]:
# Get a single group as a DataFrame
grouped.get_group('Sun').head()

## Basic Aggregation

Apply a single aggregation function to each group.

In [ ]:
# Mean of each numeric column, grouped by day
tips.groupby('day')[['total_bill', 'tip']].mean()

In [ ]:
# Sum of tips by time of day
tips.groupby('time')['tip'].sum()

In [ ]:
# Count of records per group
tips.groupby('day')['total_bill'].count()

## `.agg()` — Multiple Aggregation Functions

`.agg()` (or `.aggregate()`) lets you apply one or more functions to one or more columns in a single call.

In [ ]:
# Single function
tips.groupby('day')['total_bill'].agg('mean')

In [ ]:
# Multiple functions on a single column
tips.groupby('day')['total_bill'].agg(['mean', 'median', 'std', 'count'])

In [ ]:
# Different functions for different columns
tips.groupby('day').agg({
    'total_bill': ['mean', 'sum'],
    'tip': ['mean', 'max'],
    'size': 'mean'
})

In [ ]:
# Custom aggregation function
def tip_range(series):
    return series.max() - series.min()

tips.groupby('day')['tip'].agg(['mean', tip_range])

## Named Aggregation with `pd.NamedAgg`

Named aggregation gives you clean, flat column names in the result instead of multi-level columns.

In [ ]:
tips.groupby('day').agg(
    avg_bill=pd.NamedAgg(column='total_bill', aggfunc='mean'),
    total_tips=pd.NamedAgg(column='tip', aggfunc='sum'),
    max_tip=pd.NamedAgg(column='tip', aggfunc='max'),
    num_meals=pd.NamedAgg(column='total_bill', aggfunc='count')
)

In [ ]:
# Shorthand without pd.NamedAgg (tuple of column, function)
tips.groupby('day').agg(
    avg_bill=('total_bill', 'mean'),
    total_tips=('tip', 'sum'),
    max_party=('size', 'max')
)

## `.transform()` vs `.agg()`

| | `.agg()` | `.transform()` |
|---|---|---|
| **Returns** | One row per group (reduced) | Same number of rows as input |
| **Use case** | Summary statistics | Add group-level info back to each row |

`transform()` broadcasts the group result back to the original DataFrame shape.

In [ ]:
# agg: one value per group
tips.groupby('day')['total_bill'].agg('mean')

In [ ]:
# transform: the group mean repeated for every row in that group
tips['day_avg_bill'] = tips.groupby('day')['total_bill'].transform('mean')
tips[['day', 'total_bill', 'day_avg_bill']].head(10)

In [ ]:
# Practical use: compute how each bill deviates from its day's average
tips['bill_vs_avg'] = tips['total_bill'] - tips['day_avg_bill']
tips[['day', 'total_bill', 'day_avg_bill', 'bill_vs_avg']].head(10)

In [ ]:
# Standardise within groups (z-score per day)
tips['bill_zscore'] = tips.groupby('day')['total_bill'].transform(
    lambda x: (x - x.mean()) / x.std()
)
tips[['day', 'total_bill', 'bill_zscore']].head(10)

## Multi-column GroupBy

In [ ]:
# Group by multiple columns
tips.groupby(['day', 'time'])['total_bill'].mean()

In [ ]:
# Unstack for a cleaner view
tips.groupby(['day', 'time'])['total_bill'].mean().unstack()

In [ ]:
# Group by day, time, and smoker status
tips.groupby(['day', 'time', 'smoker']).agg(
    avg_tip=('tip', 'mean'),
    count=('tip', 'count')
).head(10)

## `.describe()` on Groups

In [ ]:
tips.groupby('day')['total_bill'].describe()

In [ ]:
tips.groupby('time')[['total_bill', 'tip']].describe()

## Clean Up

Drop the extra columns we added during exploration.

In [ ]:
tips = tips.drop(columns=['day_avg_bill', 'bill_vs_avg', 'bill_zscore'], errors='ignore')
tips.head()

## Summary

In this notebook we covered GroupBy and aggregation:

- **Split-apply-combine**: `.groupby()` splits data, a function is applied per group, results are combined.
- `.agg()` computes summary statistics (mean, sum, count, etc.) and returns one row per group.
- `.transform()` returns results broadcast back to the original shape — ideal for adding group-level features.
- Named aggregation (`pd.NamedAgg` or tuple syntax) produces clean, flat column names.
- Multi-column groupby creates hierarchical groupings; use `.unstack()` to reshape.
- `.describe()` on a GroupBy object gives full descriptive statistics per group.

Next up: **Combining DataFrames** — concat, merge, and join.